# 01_18 — Avaliação mínima de exequibilidade de um projeto ABM/MBA com apoio de LLM

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path
import os
import sys
import platform
import datetime

HOME = Path.home()
CANDIDATAS_TRABALHO = [HOME / 'work', HOME / 'laboratorio', Path('/home/jovyan/work'), Path('/home/jovyan/laboratorio')]
PASTA_TRABALHO = None
for pasta in CANDIDATAS_TRABALHO:
    if pasta.exists() and os.access(pasta, os.W_OK):
        PASTA_TRABALHO = pasta
        break
if PASTA_TRABALHO is None:
    PASTA_TRABALHO = Path.cwd()
PASTA_SAIDA = PASTA_TRABALHO / 'avaliacoes_exequibilidade'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0])
print('Sistema:', platform.platform())
print('Pasta de trabalho escolhida:', PASTA_TRABALHO)
print('Pasta de saída:', PASTA_SAIDA)

In [ ]:
nome_do_aluno = 'vinicius'
projeto = {
    'titulo': 'Simulação de Difusão de Inovação em Rede Social Simples',
    'area_ou_tema': 'Adoção de tecnologia / Opinião pública',
    'ideia_em_poucas_linhas': 'A simulação visa observar como um novo aplicativo (ou ideia) se espalha em uma população.',
    'pergunta_principal': 'Quanto tempo demora para uma inovação atingir 80% da população com base na taxa de persuasão?',
    'agentes_imaginados': 'Pessoas (indivíduos de uma comunidade).',
    'tipos_de_agentes': 'Há dois status para o mesmo agente: Não Adotantes e Adotantes.',
    'ambiente': 'População em rede simples.',
    'regras_de_comportamento': 'Se o agente já é Adotante, ele tenta convencer vizinhos.',
    'interacoes': 'Os agentes interagem apenas com vizinhos mais próximos.',
    'parametros': 'Número total de agentes, porcentagem inicial de adotantes e probabilidade de contágio.',
    'resultados_esperados': 'Um gráfico de linha mostrando o crescimento do número de Adotantes ao longo do tempo.',
    'tamanho_desejado': '100 agentes, 50 passos, repetido 10 vezes.',
    'nivel_de_experiencia': 'iniciante',
    'preocupacoes': 'Gerar o código inicial da lógica de transição de estados e plotar o gráfico.'
}

In [ ]:
import json
import os
import re

def criar_slug(texto, limite=60):
    texto = texto.lower().strip()
    texto = re.sub(r'[^a-z0-9áàãâéêíóôõúç_-]+', '_', texto)
    texto = texto.strip('_')
    if not texto:
        texto = 'projeto'
    return texto[:limite]

def montar_prompt_exequibilidade(projeto, ambiente):
    projeto_json = json.dumps(projeto, ensure_ascii=False, indent=2)
    ambiente_json = json.dumps(ambiente, ensure_ascii=False, indent=2)
    prompt = f'''
Você é um assistente didático...
Tarefa: avalie a exequibilidade mínima do projeto.
Descrição do projeto: {projeto_json}
Informações do ambiente: {ambiente_json}
Responda em português, com linguagem clara para iniciante.
'''
    return prompt

ambiente_simserv = {
    'python': sys.version.split()[0],
    'sistema': platform.platform(),
    'pasta_trabalho': str(PASTA_TRABALHO),
    'pasta_saida': str(PASTA_SAIDA),
    'bibliotecas': ['numpy', 'pandas', 'matplotlib', 'networkx', 'mesa', 'requests', 'psutil']
}

prompt_exequibilidade = montar_prompt_exequibilidade(projeto, ambiente_simserv)
arquivo_prompt = PASTA_SAIDA / f'prompt_exequibilidade_{criar_slug(projeto["titulo"])}.txt'
arquivo_prompt.write_text(prompt_exequibilidade, encoding='utf-8')
print('Prompt salvo em:', arquivo_prompt)

In [ ]:
import os
import requests

base_url_ollama = os.getenv('OLLAMA_BASE_URL', 'https://api.ollama.com').rstrip('/')
api_key = os.getenv('OLLAMA_API_KEY')
modelo_escolhido = os.getenv('OLLAMA_MODEL', 'qwen3-next:80b')
resposta_llm = None
erro_llm = None

headers = {}
if api_key:
    headers['Authorization'] = f'Bearer {api_key}'

urls = [base_url_ollama]
if 'http://localhost:11434' not in urls:
    urls.append('http://localhost:11434')
if 'http://127.0.0.1:11434' not in urls:
    urls.append('http://127.0.0.1:11434')
if 'http://ollama:11434' not in urls:
    urls.append('http://ollama:11434')

endpoint_encontrado = None
for url in urls:
    try:
        resp = requests.get(f'{url}/api/tags', headers=headers, timeout=15)
        if resp.ok:
            try:
                data = resp.json()
                if 'models' in data:
                    endpoint_encontrado = url
                    break
            except Exception:
                pass
    except Exception:
        pass

base_url_ollama = endpoint_encontrado
if base_url_ollama is None:
    resposta_llm = f'Não foi possível consultar a Ollama Cloud. Prompt salvo em: {arquivo_prompt}'
    erro_llm = 'Endpoint Ollama não encontrado.'
else:
    try:
        payload = {
            'model': modelo_escolhido,
            'messages': [{'role': 'user', 'content': prompt_exequibilidade}],
            'stream': False,
            'options': {'temperature': 0.2},
        }
        resp = requests.post(
            f'{base_url_ollama}/api/chat',
            headers=headers,
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        resposta_llm = resp.json().get('message', {}).get('content', '').strip()
        if not resposta_llm:
            resposta_llm = resp.json().get('response', '').strip()
    except Exception as exc:
        erro_llm = f'Falha ao consultar o Ollama em {base_url_ollama}: {exc}'
        resposta_llm = f'Não foi possível obter uma resposta da IA.\nErro: {erro_llm}\nPrompt salvo em: {arquivo_prompt}'

print('base_url_ollama:', base_url_ollama)
print('modelo_escolhido:', modelo_escolhido)
print(resposta_llm[:2000] if resposta_llm else '')

In [ ]:
import datetime
import re

slug = re.sub(r'[^a-z0-9]+', '_', (projeto.get('titulo', 'projeto') or 'projeto').lower()).strip('_')[:60] or 'projeto'
data_hora = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
arquivo_relatorio_md = PASTA_SAIDA / f'relatorio_exequibilidade_{slug}_{data_hora}.md'
conteudo = f'''# Relatório de exequibilidade mínima

Aluno: {nome_do_aluno}
Projeto: {projeto.get('titulo', '')}
Data/hora: {datetime.datetime.now()}
Modelo LLM: {modelo_escolhido}
Base URL Ollama: {base_url_ollama}

---

{resposta_llm}
'''
arquivo_relatorio_md.write_text(conteudo, encoding='utf-8')
print(arquivo_relatorio_md)
print('Arquivo existe:', arquivo_relatorio_md.exists())